# DSPy — Otimização com `GEPA`

Neste notebook será demonstrado o uso do otimizador `GEPA` do DSPy em um problema de **classificação binária de textos**.

Será utilizada a base **Natural Language Processing with Disaster Tweets**, disponibilizada no Kaggle. O objetivo é classificar cada tweet em uma das seguintes categorias:

* `0`: o tweet **não descreve um desastre real**;
* `1`: o tweet **descreve um desastre real**.

O experimento será dividido em duas etapas:

1. avaliar um classificador DSPy utilizando a instrução original definida na `Signature`;
2. utilizar o `GEPA` para evoluir reflexivamente a instrução do programa a partir das execuções, dos erros observados e do feedback fornecido pela métrica.

`GEPA` significa **Genetic-Pareto**. Trata-se de um otimizador evolutivo e reflexivo que utiliza um modelo de linguagem separado para analisar os resultados do programa e propor novas instruções.

Diferentemente do `COPRO`, que explora instruções candidatas por otimização coordenada, o `GEPA` utiliza **reflexão sobre traces de execução**, **feedback textual** e **seleção baseada em uma fronteira de Pareto** para orientar a evolução dos prompts.

Neste experimento, os dados serão separados em:

* **treino**, utilizado nas atualizações reflexivas;
* **validação**, utilizada para acompanhar o desempenho dos candidatos e selecionar o programa final;
* **teste**, mantido separado para a avaliação final.

A avaliação final continuará utilizando:

* Accuracy;
* Precision;
* Recall;
* F1-score.

A métrica principal para comparar o programa original e o programa otimizado será o **F1-score**.

In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env
import dspy  # Framework para otimização de prompts com Language Models
import pandas as pd

from typing import Literal
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm

/home/leonardo/Documentos/github/dspy_studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()  # Lê variáveis do arquivo .env

True

## Setup - Configuração do Modelo

Primeiro, carregamos as variáveis de ambiente (como API key) do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

In [3]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo utilizado pelo classificador
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Modelo utilizado pelo GEPA para refletir sobre erros e propor novas instruções.
# A documentação recomenda utilizar um modelo forte para essa etapa.
reflection_lm = dspy.LM(
    "openai/gpt-5-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
    max_tokens=32000,
)

# Configura o modelo do classificador como LM padrão do DSPy
dspy.configure(lm=lm)

## 3. Leitura da base de dados

Será utilizada a base da competição **Natural Language Processing with Disaster Tweets**, do Kaggle.

Referência:

https://www.kaggle.com/competitions/nlp-getting-started

Para este experimento são relevantes principalmente duas colunas:

| Coluna   | Descrição                   |
| -------- | --------------------------- |
| `text`   | Texto do tweet              |
| `target` | Classe correta (`0` ou `1`) |

O problema consiste, portanto, em aprender a relação:

`text → target`


In [4]:
df = pd.read_csv('disaster_tweets.csv')
df = df.head(200)
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Análise da distribuição das classes

Antes da divisão dos dados, é importante verificar quantos exemplos existem de cada classe.

Além da quantidade absoluta, será analisada a proporção entre tweets classificados como `0` e `1`.

Essa análise é importante porque o **F1-score** considera conjuntamente precisão e recall e é especialmente útil quando existe algum grau de desbalanceamento entre as classes.


In [5]:
df["target"].value_counts()

target
0    103
1     97
Name: count, dtype: int64

In [6]:
df["target"].value_counts(normalize=True)

target
0    0.515
1    0.485
Name: proportion, dtype: float64

## Separação entre treino, validação e teste

Para utilizar o `GEPA` de forma adequada, a base será dividida em três conjuntos:

* **70% para treino**;
* **15% para validação**;
* **15% para teste**.

O parâmetro `stratify` é utilizado para manter aproximadamente a mesma proporção entre as classes `0` e `1` em todos os conjuntos.

No `GEPA`, os conjuntos de treino e validação possuem funções diferentes:

* o **trainset** fornece os exemplos utilizados nas atualizações reflexivas da instrução;
* o **valset** é utilizado para acompanhar as pontuações dos candidatos na fronteira de Pareto e selecionar o programa retornado pela otimização;
* o **testset** não participa da otimização e é utilizado somente na comparação final entre o baseline e o programa otimizado.

Essa separação evita que o conjunto utilizado para medir a generalização final também seja utilizado na escolha das instruções.

In [7]:
# Primeiro separamos 70% para treino e 30% para validação + teste
df_train, df_temp = train_test_split(
    df[["text", "target"]],
    test_size=0.30,
    random_state=42,
    stratify=df["target"],
)

# Divide os 30% restantes igualmente: 15% validação e 15% teste
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["target"],
)

print(f"Treino:     {len(df_train)} exemplos")
print(f"Validação:  {len(df_val)} exemplos")
print(f"Teste:      {len(df_test)} exemplos")

Treino:     140 exemplos
Validação:  30 exemplos
Teste:      30 exemplos


## Conversão para `dspy.Example`

O DSPy representa exemplos de treino e teste por meio da classe `dspy.Example`.

Neste problema, cada exemplo possui dois campos:

* `text`: entrada fornecida ao modelo;
* `target`: resposta esperada.

A chamada:

`with_inputs("text")`

informa explicitamente ao DSPy que `text` deve ser utilizado como entrada do programa.

Consequentemente, `target` passa a ser tratado como o **label**, ou seja, a resposta esperada para aquele exemplo.

Conceitualmente, cada registro passa a ter a seguinte estrutura:

`entrada: text → saída esperada: target`

In [8]:
def dataframe_para_dspy(dataframe):
    exemplos = []

    for _, row in dataframe.iterrows():
        exemplo = dspy.Example(
            text=row["text"],
            target=int(row["target"]),
        ).with_inputs("text")

        exemplos.append(exemplo)

    return exemplos

In [9]:
trainset = dataframe_para_dspy(df_train)
valset = dataframe_para_dspy(df_val)
testset = dataframe_para_dspy(df_test)

print(f"Trainset DSPy: {len(trainset)}")
print(f"Valset DSPy:   {len(valset)}")
print(f"Testset DSPy:  {len(testset)}")

Trainset DSPy: 140
Valset DSPy:   30
Testset DSPy:  30


In [10]:
trainset[0]

Example({'text': '13,000 people receive #wildfires evacuation orders in California ', 'target': 1}) (input_keys={'text'})

## Definição da tarefa com uma `Signature`

No DSPy, uma `Signature` descreve declarativamente a tarefa que será executada pelo modelo.

A `ClassificarTweet` possui:

* um `InputField` chamado `text`, contendo o tweet;
* um `OutputField` chamado `target`, contendo a classificação.

O tipo:

`Literal[0, 1]`

restringe a resposta esperada às duas classes válidas do problema.

Dessa forma, a Signature define claramente o contrato:

`texto do tweet → 0 ou 1`


In [11]:
class ClassificarTweet(dspy.Signature):
    """
    Determine se o tweet descreve um desastre real.

    Retorne:
    - 1 se o tweet estiver relacionado a um desastre real.
    - 0 caso contrário.
    """

    text: str = dspy.InputField(
        desc="Texto do tweet que deve ser classificado."
    )

    target: Literal[0, 1] = dspy.OutputField(
        desc="1 para desastre real e 0 para não desastre."
    )

In [12]:
classificador_base = dspy.Predict(ClassificarTweet)

In [13]:
def avaliar_classificador(programa, dataset, descricao="Avaliando"):
    """
    Executa um programa DSPy sobre um dataset e calcula
    métricas globais de classificação.
    """

    y_true = []
    y_pred = []

    for exemplo in tqdm(dataset, desc=descricao):

        # Executa o programa utilizando somente os campos
        # marcados como entrada pelo with_inputs(...)
        predicao = programa(**exemplo.inputs())

        # Label verdadeiro
        y_true.append(int(exemplo.target))

        # Label previsto pelo DSPy
        y_pred.append(int(predicao.target))

    # Calcula as métricas sobre todo o conjunto
    resultado = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

    return resultado

In [14]:
resultado_base = avaliar_classificador(
    classificador_base,
    testset,
    descricao="Baseline zero-shot",
)

Baseline zero-shot: 100%|█| 30/30 [00:0


## Avaliação do baseline

O classificador inicial será executado sobre todos os exemplos do conjunto de teste.

Para cada exemplo:

1. o campo `text` é enviado ao programa;
2. o programa produz uma previsão para `target`;
3. a previsão é comparada com o `target` verdadeiro.

Ao final são calculadas as métricas globais de classificação.

Esse resultado será considerado o desempenho **antes da otimização**.


In [15]:
print(f"F1 baseline: {resultado_base['f1']:.4f}")

F1 baseline: 0.9677


In [16]:
print(f"Accuracy:  {resultado_base['accuracy']:.4f}")
print(f"Precision: {resultado_base['precision']:.4f}")
print(f"Recall:    {resultado_base['recall']:.4f}")
print(f"F1:        {resultado_base['f1']:.4f}")

Accuracy:  0.9667
Precision: 0.9375
Recall:    1.0000
F1:        0.9677


## Métrica utilizada durante a otimização

O `GEPA` utiliza uma métrica para avaliar as previsões dos candidatos, mas possui uma característica importante: além de uma pontuação numérica, a métrica pode retornar **feedback textual**.

Esse feedback é fornecido ao modelo de reflexão e ajuda o otimizador a entender **por que** uma previsão falhou.

Neste problema, cada previsão receberá:

* `score=1.0` quando a classificação estiver correta;
* `score=0.0` quando a classificação estiver incorreta.

Nos erros, também será informado se ocorreu um **falso positivo** ou um **falso negativo**.

```python
def metrica_gepa(example, prediction, trace=None, pred_name=None, pred_trace=None):
    ...
    return dspy.Prediction(
        score=score,
        feedback=feedback,
    )
```

O F1-score continuará sendo calculado globalmente no conjunto de teste. A métrica do GEPA, por outro lado, é aplicada individualmente aos exemplos durante a otimização.

In [17]:
def metrica_gepa(
    example,
    prediction,
    trace=None,
    pred_name=None,
    pred_trace=None,
):
    """
    Métrica utilizada internamente pelo GEPA.

    Retorna uma pontuação numérica e, em caso de erro,
    um feedback textual para orientar a reflexão do otimizador.
    """
    esperado = int(example.target)
    previsto = int(prediction.target)

    # Classificação correta
    if esperado == previsto:
        return dspy.Prediction(
            score=1.0,
            feedback=None,
        )

    # Classificação incorreta: fornece informação adicional ao GEPA
    if esperado == 1 and previsto == 0:
        tipo_erro = "Falso negativo"
    else:
        tipo_erro = "Falso positivo"

    feedback = (
        f"{tipo_erro}: o rótulo correto é {esperado}, "
        f"mas o programa retornou {previsto}. "
        "Analise o tweet e ajuste a instrução para distinguir melhor "
        "desastres reais de usos figurados, comentários ou situações não catastróficas."
    )

    return dspy.Prediction(
        score=0.0,
        feedback=feedback,
    )

## Otimização com `GEPA`

O `GEPA` (**Genetic-Pareto**) é um otimizador evolutivo e reflexivo do DSPy.

Em vez de simplesmente gerar variações independentes de uma instrução, o `GEPA` analisa **traces de execução**, pontuações e feedback textual para propor mudanças direcionadas no programa.

Neste notebook, como o programa possui apenas um `dspy.Predict`, o principal componente textual otimizado será a instrução da `Signature`.

O funcionamento pode ser representado conceitualmente como:

```text
programa original
       ↓
execução em exemplos do trainset
       ↓
score + feedback textual
       ↓
reflexão sobre erros e traces
       ↓
proposta de uma nova instrução
       ↓
avaliação do novo candidato
       ↓
atualização da fronteira de Pareto
       ↓
novas mutações e possíveis combinações
       ↓
melhor candidato no valset
```

A seleção baseada em Pareto ajuda o otimizador a preservar candidatos que funcionam bem em diferentes exemplos, em vez de manter apenas uma única trajetória de otimização.

In [18]:
AUTO = "light"

optimizer = dspy.GEPA(
    metric=metrica_gepa,                   # Score + feedback textual da tarefa
    reflection_lm=reflection_lm,           # LM responsável pela reflexão e pelas novas instruções
    auto=AUTO,                             # Orçamento automático: "light", "medium" ou "heavy"
    reflection_minibatch_size=3,           # Exemplos utilizados em cada etapa de reflexão
    candidate_selection_strategy="pareto", # Seleção de candidatos pela fronteira de Pareto
    num_threads=4,                         # Paralelismo das avaliações
    track_stats=True,                      # Mantém resultados detalhados da otimização
    seed=42,                               # Reprodutibilidade
)

## Compilação do programa

A compilação do `GEPA` recebe:

* o programa que será otimizado (`student`);
* o `trainset`, utilizado nas atualizações reflexivas;
* o `valset`, utilizado para acompanhar as pontuações dos candidatos e escolher o programa final.

```python
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
    valset=valset,
)
```

Ao contrário do notebook de `COPRO`, não é necessário passar `eval_kwargs` ao `compile()`. O paralelismo é configurado diretamente por `num_threads` no construtor do `GEPA`.

In [19]:
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
    valset=valset,
)

2026/09/07 21:32:31 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 500 metric calls of the program. This amounts to 2.94 full evals on the train+val set.
2026/09/07 21:32:31 INFO dspy.teleprompt.gepa.gepa: Using 30 examples for tracking Pareto scores.
GEPA Optimization:   0%| | 0/500 [00:002026/09/07 21:32:31 INFO dspy.evaluate.evaluate: Average Metric: 29.0 / 30 (96.7%)
2026/09/07 21:32:31 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.9666666666666667 over 30 / 30 examples
2026/09/07 21:32:31 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.9666666666666667


Average Metric: 3.00 / 3 (100.0%): 100%

2026/09/07 21:32:31 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/09/07 21:32:31 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2026/09/07 21:32:31 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
2026/09/07 21:32:31 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 0.9666666666666667



Average Metric: 3.00 / 3 (100.0%): 100%

2026/09/07 21:32:31 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/09/07 21:32:31 INFO dspy.teleprompt.gepa.gepa: Iteration 2: All subsample scores perfect. Skipping.
2026/09/07 21:32:31 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Reflective mutation did not propose a new candidate
GEPA Optimization:   7%| | 36/500 [00:02026/09/07 21:32:31 INFO dspy.teleprompt.gepa.gepa: Iteration 3: Selected program 0 score: 0.9666666666666667


...


Average Metric: 3.00 / 3 (100.0%): 100%

2026/09/07 22:08:46 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/09/07 22:08:46 INFO dspy.teleprompt.gepa.gepa: Iteration 53: All subsample scores perfect. Skipping.
2026/09/07 22:08:46 INFO dspy.teleprompt.gepa.gepa: Iteration 53: Reflective mutation did not propose a new candidate
GEPA Optimization:  97%|▉| 486/500 [36:2026/09/07 22:08:46 INFO dspy.teleprompt.gepa.gepa: Iteration 54: Selected program 9 score: 0.9666666666666667



Average Metric: 2.00 / 3 (66.7%): 100%|

2026/09/07 22:09:03 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 3 (66.7%)


2026/09/07 22:09:29 INFO dspy.teleprompt.gepa.gepa: Iteration 54: Proposed new text for self: Tarefa (resumida)
Classificar um tweet como descrevendo um "desastre real" (retornar "1") ou NÃO (retornar "0").

Saída obrigatória (regra inviolável)
- Responder estritamente com um único carácter ASCII: "1" ou "0".
- Não retornar nada além desse carácter — sem explicações, sem espaços, sem quebras de linha, sem pontuação extra.

Definição operacional de "desastre real"
Um tweet descreve um desastre real quando relata, reporta ou descreve um evento adverso concreto e factual com relevância pública ou gravidade evidente — por exemplo: terremotos, tsunamis, enchentes, incêndios grandes, colisões/accidentes rodoviários/ferroviários/aéreos, explosões, desabamentos, ataques violentos com feridos, grandes vazamentos químicos, surtos/epidemias reais, manifestações violentas com feridos, evacuação em massa, ou qualquer incidente que claramente envolva vítimas múltiplas, mortes, feridos sérios, danos 

## Inspeção do resultado da otimização

Com `track_stats=True`, o programa retornado pelo `GEPA` possui o atributo `detailed_results`.

Ele permite inspecionar, entre outras informações:

1. os candidatos gerados;
2. a linhagem (`parents`) de cada candidato;
3. a pontuação agregada de cada candidato no conjunto de validação;
4. quantas chamadas à métrica foram consumidas;
5. qual candidato foi selecionado como o melhor.

Também podemos comparar diretamente a instrução original da `Signature` com a instrução produzida pelo candidato escolhido.

In [20]:
print("=== INSTRUÇÃO ORIGINAL ===")
print(classificador_base.signature.instructions)

print("\n=== INSTRUÇÃO OTIMIZADA PELO GEPA ===")
print(classificador_otimizado.signature.instructions)

=== INSTRUÇÃO ORIGINAL ===
Determine se o tweet descreve um desastre real.

Retorne:
- 1 se o tweet estiver relacionado a um desastre real.
- 0 caso contrário.

=== INSTRUÇÃO OTIMIZADA PELO GEPA ===
Determine se o tweet descreve um desastre real.

Retorne:
- 1 se o tweet estiver relacionado a um desastre real.
- 0 caso contrário.


In [21]:
resultados_gepa = classificador_otimizado.detailed_results

print(f"Melhor candidato: {resultados_gepa.best_idx}")
print(f"Número de candidatos: {len(resultados_gepa.candidates)}")
print(f"Total de chamadas à métrica: {resultados_gepa.total_metric_calls}")
print(f"Avaliações completas do valset: {resultados_gepa.num_full_val_evals}")

Melhor candidato: 0
Número de candidatos: 11
Total de chamadas à métrica: 522
Avaliações completas do valset: 11


In [22]:
historico_gepa = pd.DataFrame(
    {
        "candidato": range(len(resultados_gepa.candidates)),
        "score_validacao": resultados_gepa.val_aggregate_scores,
        "parents": resultados_gepa.parents,
        "metric_calls_ate_descoberta": resultados_gepa.discovery_eval_counts,
    }
).sort_values(
    "score_validacao",
    ascending=False,
)

historico_gepa

,candidato,score_validacao,parents,metric_calls_ate_descoberta
0,0,0.966667,[None],0
1,1,0.966667,[0],42
2,2,0.966667,[1],87
6,6,0.966667,[5],288
5,5,0.966667,[2],222
9,9,0.966667,[6],426
4,4,0.933333,[2],177
3,3,0.933333,[2],126
7,7,0.933333,[6],333
8,8,0.933333,[6],378


In [23]:
resultado_otimizado = avaliar_classificador(
    classificador_otimizado,
    testset,
    descricao="GEPA",
)

GEPA: 100%|█| 30/30 [00:00<00:00, 65.74


## Avaliação após a aplicação do `GEPA`

O programa otimizado pelo `GEPA` será avaliado utilizando **exatamente o mesmo conjunto de teste utilizado pelo baseline**.

Isso permite comparar:

* o classificador utilizando a instrução original;
* o classificador utilizando a instrução selecionada pelo `GEPA`.

O `testset` não foi utilizado nem nas atualizações reflexivas nem na seleção dos candidatos.

Durante a otimização:

* o `trainset` forneceu os exemplos usados para reflexão;
* o `valset` foi utilizado para avaliar os candidatos e selecionar o programa final.

Ao final serão calculados novamente:

* Accuracy;
* Precision;
* Recall;
* F1-score.

Embora o `GEPA` utilize uma métrica por exemplo com score e feedback textual, o **F1-score global** continua sendo a métrica principal para comparar o programa original com o programa otimizado no conjunto de teste.

In [24]:
comparacao = pd.DataFrame(
    {
        "Modelo": [
            "Baseline (instrução original)",
            "GEPA",
        ],
        "Accuracy": [
            resultado_base["accuracy"],
            resultado_otimizado["accuracy"],
        ],
        "Precision": [
            resultado_base["precision"],
            resultado_otimizado["precision"],
        ],
        "Recall": [
            resultado_base["recall"],
            resultado_otimizado["recall"],
        ],
        "F1": [
            resultado_base["f1"],
            resultado_otimizado["f1"],
        ],
    }
)

comparacao

,Modelo,Accuracy,Precision,Recall,F1
0,Baseline (instrução original),0.966667,0.9375,1.0,0.967742
1,GEPA,0.966667,0.9375,1.0,0.967742


In [25]:
print("BASELINE")
print(
    classification_report(
        resultado_base["y_true"],
        resultado_base["y_pred"],
        digits=4,
    )
)

BASELINE
              precision    recall  f1-score   support

           0     1.0000    0.9333    0.9655        15
           1     0.9375    1.0000    0.9677        15

    accuracy                         0.9667        30
   macro avg     0.9688    0.9667    0.9666        30
weighted avg     0.9688    0.9667    0.9666        30



In [26]:
print(f"GEPA (auto={AUTO})")

print(
    classification_report(
        resultado_otimizado["y_true"],
        resultado_otimizado["y_pred"],
        digits=4,
    )
)

GEPA (auto=light)
              precision    recall  f1-score   support

           0     1.0000    0.9333    0.9655        15
           1     0.9375    1.0000    0.9677        15

    accuracy                         0.9667        30
   macro avg     0.9688    0.9667    0.9666        30
weighted avg     0.9688    0.9667    0.9666        30



## Salvando o programa otimizado pelo `GEPA`

Neste experimento, o programa possui uma arquitetura simples baseada em `dspy.Predict`, e o `GEPA` modifica principalmente o estado aprendido da `Signature`, especialmente sua instrução.

Por isso, será utilizado o **State-only Saving** do DSPy.

```python
classificador_otimizado.save("GEPA.json")
```

Esse formato salva o estado otimizado em JSON, mas não a estrutura Python completa do programa.

Para carregar o arquivo posteriormente, é necessário recriar a mesma arquitetura (`dspy.Predict(ClassificarTweet)`) e então aplicar `.load()`.

O objeto `detailed_results`, utilizado para analisar todo o processo evolutivo do GEPA durante esta execução, não é necessário para executar o classificador otimizado.

In [27]:
classificador_otimizado.save("GEPA.json")

In [28]:
# Recria a mesma arquitetura do programa
classificador_carregado = dspy.Predict(ClassificarTweet)

# Carrega o estado otimizado pelo GEPA
classificador_carregado.load("GEPA.json")

classificador_carregado

Predict(StringSignature(text -> target
    instructions='Determine se o tweet descreve um desastre real.\n\nRetorne:\n- 1 se o tweet estiver relacionado a um desastre real.\n- 0 caso contrário.'
    text = Field(annotation=str required=True json_schema_extra={'desc': 'Texto do tweet que deve ser classificado.', '__dspy_field_type': 'input', 'prefix': 'Text:'})
    target = Field(annotation=Literal[0, 1] required=True json_schema_extra={'desc': '1 para desastre real e 0 para não desastre.', '__dspy_field_type': 'output', 'prefix': 'Target:'})
))

In [29]:
tweet = "My phone battery died right before the meeting, what a disaster!"

predicao = classificador_carregado(
    text=tweet
)

print(predicao)

Prediction(
    target=0
)


In [30]:
# Mostra última chamada ao modelo (n=1 significa 1 última chamada)
dspy.inspect_history(n=1)





[2026-09-07T22:11:33.239084]

System message:

Your input fields are:
1. `text` (str): Texto do tweet que deve ser classificado.
Your output fields are:
1. `target` (Literal[0, 1]): 1 para desastre real e 0 para não desastre.
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "target": "{target}        # note: the value you produce must exactly match (no extra characters) one of: 0; 1"
}
In adhering to this structure, your objective is: 
        Determine se o tweet descreve um desastre real.
        
        Retorne:
        - 1 se o tweet estiver relacionado a um desastre real.
        - 0 caso contrário.


User message:

[[ ## text ## ]]
My phone battery died right before the meeting, what a disaster!

Respond with a JSON object in the following order of fields: `target` (must be formatted as a valid Pyth